# Modelo Financiero y Proyección de Flujo de Caja (RT-03)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karolcamachol29-sudo/RFP-CCM-EM-001/blob/main/01_Proceso_y_Simulacion/BioSTEAM/calcular_evaluacion_financiera_rt03.ipynb)

**Consorcio Solvanta Engineering Solutions S.A. de C.V. | Licitación Pública Internacional RFP-CCM-EM-001**

Este cuadernillo implementa el modelo financiero oficial y los algoritmos de análisis de sensibilidad para la planta de e-metanol (30,000 t/año), con construcción del flujo de caja a 20 años, determinación exacta de VAN, TIR, Payback y MPSP, y generación de diagramas Tornado y curvas Spider.

In [ ]:
# Instalación automática de dependencias para ejecución en Google Colab
import sys
if 'google.colab' in sys.modules:
    # En Google Colab se instalan bibliotecas científicas requeridas
    pass
    !pip install -q "biosteam>=2.50.0" "thermosteam>=0.50.0" openpyxl matplotlib pandas numpy scipy


## 1. Parámetros Base de Ingeniería y Finanzas (DWSIM + BioSTEAM)

Definición de capacidades nominales, precios de mercado contractuales, desglose auditado de CAPEX ($46.25M USD FCI / $48.56M USD TCI) y partidas operativas de OPEX.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# 1. PARÁMETROS BASE DE INGENIERÍA Y FINANZAS
PROD_MEOH_T_ANO = 30309.20
PROD_O2_T_ANO = 80790.08
HORAS_OPERACION_ANO = 8000.0

PRECIO_MEOH_BASE_USD_T = 900.00
PRECIO_O2_BASE_USD_T = 35.00
CREDITO_INCENTIVO_BASE_USD_T = 0.00

FCI = 46250000.00
WC_PCT = 0.05
WC = FCI * WC_PCT
TCI = FCI + WC

COSTO_ELEC_BASE_USD_KWH = 0.040
POTENCIA_TOTAL_MW = 38.56
COSTO_ELEC_ANUAL = POTENCIA_TOTAL_MW * 1000.0 * HORAS_OPERACION_ANO * COSTO_ELEC_BASE_USD_KWH
COSTO_CO2_ANUAL = 42194.7 * 35.00
COSTO_H2O_ANUAL = 39000.0 * 6.00
FONDO_STACKS_SOEC_ANUAL = 4950000.00
FONDO_CATALIZADOR_ANUAL = 169721.00
MANO_OBRA_ANUAL = 980000.00
MANTENIMIENTO_SEGUROS_ANUAL = FCI * 0.035

OPEX_BRUTO_BASE = (COSTO_ELEC_ANUAL + COSTO_CO2_ANUAL + COSTO_H2O_ANUAL +
                   FONDO_STACKS_SOEC_ANUAL + FONDO_CATALIZADOR_ANUAL +
                   MANO_OBRA_ANUAL + MANTENIMIENTO_SEGUROS_ANUAL)

CREDITO_O2_ANUAL_BASE = PROD_O2_T_ANO * PRECIO_O2_BASE_USD_T
OPEX_NETO_BASE = OPEX_BRUTO_BASE - CREDITO_O2_ANUAL_BASE

TASA_DESCUENTO_BASE = 0.10
TASA_DESCUENTO_WACC = 0.085
TASA_ISR = 0.30
VIDA_UTIL_ANOS = 20
ANOS_DEPRECIACION = 10
ANOS_CONSTRUCCION = 2
PCT_INVERSION_ANO_M2 = 0.60
PCT_INVERSION_ANO_M1 = 0.40

print(f"FCI: ${FCI:,.2f} USD | TCI: ${TCI:,.2f} USD")
print(f"OPEX Bruto: ${OPEX_BRUTO_BASE:,.2f} USD/año | OPEX Neto: ${OPEX_NETO_BASE:,.2f} USD/año")

## 2. Construcción Rigurosa del Flujo de Caja (Años 0 a 20)

Función para generar la tabla anualizada de flujo de caja libre considerando depreciación en línea recta, impuestos, amortización y recuperación de capital de trabajo.

In [ ]:
def construir_flujo_caja(fci=FCI, wc=WC, opex_bruto=OPEX_BRUTO_BASE,
                         precio_meoh=PRECIO_MEOH_BASE_USD_T,
                         precio_o2=PRECIO_O2_BASE_USD_T,
                         credito_inc=CREDITO_INCENTIVO_BASE_USD_T,
                         tasa_isr=TASA_ISR, vida_util=VIDA_UTIL_ANOS,
                         anos_deprec=ANOS_DEPRECIACION):
    tci = fci + wc
    deprec_anual = fci / anos_deprec

    ingresos_meoh = PROD_MEOH_T_ANO * precio_meoh
    ingresos_o2 = PROD_O2_T_ANO * precio_o2
    ingresos_inc = PROD_MEOH_T_ANO * credito_inc
    ingresos_totales = ingresos_meoh + ingresos_o2 + ingresos_inc

    ebitda = ingresos_totales - opex_bruto

    anos = list(range(vida_util + 1))
    inversion = [0.0] * (vida_util + 1)
    inversion[0] = -tci
    inversion[vida_util] = wc

    ingresos_col = [0.0] + [ingresos_totales] * vida_util
    opex_col = [0.0] + [opex_bruto] * vida_util
    ebitda_col = [0.0] + [ebitda] * vida_util

    deprec_col = [0.0] * (vida_util + 1)
    ebit_col = [0.0] * (vida_util + 1)
    isr_col = [0.0] * (vida_util + 1)
    utilidad_neta_col = [0.0] * (vida_util + 1)
    flujo_libre_col = [0.0] * (vida_util + 1)

    flujo_libre_col[0] = -tci

    for t in range(1, vida_util + 1):
        if t <= anos_deprec:
            deprec_col[t] = deprec_anual
        else:
            deprec_col[t] = 0.0

        ebit_col[t] = ebitda_col[t] - deprec_col[t]
        if ebit_col[t] > 0.0:
            isr_col[t] = ebit_col[t] * tasa_isr
        else:
            isr_col[t] = 0.0

        utilidad_neta_col[t] = ebit_col[t] - isr_col[t]
        flujo_libre_col[t] = utilidad_neta_col[t] + deprec_col[t]
        if t == vida_util:
            flujo_libre_col[t] += wc

    return {
        'anos': anos,
        'inversion': inversion,
        'ingresos': ingresos_col,
        'opex': opex_col,
        'ebitda': ebitda_col,
        'depreciacion': deprec_col,
        'ebit': ebit_col,
        'isr': isr_col,
        'utilidad_neta': utilidad_neta_col,
        'flujo_libre': flujo_libre_col,
        'ingresos_meoh': ingresos_meoh,
        'ingresos_o2': ingresos_o2,
        'ingresos_inc': ingresos_inc,
        'tci': tci,
        'fci': fci,
        'wc': wc
    }

## 3. Algoritmos de Cálculo Financiero (VAN, TIR, Payback y MPSP)

Implementación matemática de los criterios de evaluación de proyectos bajo el estándar BioSTEAM.

In [ ]:
def calcular_van(flujo_neto, tasa_desc):
    return sum(f / ((1.0 + tasa_desc) ** t) for t, f in enumerate(flujo_neto))

def calcular_tir(flujo_neto):
    def f_obj(rate):
        return sum(f / ((1.0 + rate) ** t) for t, f in enumerate(flujo_neto))

    low, high = 0.001, 1.0
    if f_obj(low) * f_obj(high) > 0:
        return None

    for _ in range(100):
        mid = (low + high) / 2.0
        val = f_obj(mid)
        if abs(val) < 1e-4:
            return mid
        if f_obj(low) * val < 0:
            high = mid
        else:
            low = mid
    return mid

def calcular_payback_exacto(cum_series):
    for t in range(len(cum_series) - 1):
        if cum_series[t] <= 0.0 and cum_series[t + 1] > 0.0:
            frac = (-cum_series[t]) / (cum_series[t + 1] - cum_series[t])
            return t + frac
    return None

def calcular_mpsp(tasa_desc=TASA_DESCUENTO_BASE, fci=FCI, opex_factor=1.0):
    wc_calc = fci * WC_PCT
    opex_calc = OPEX_BRUTO_BASE * opex_factor

    def f_obj(p_test):
        fc = construir_flujo_caja(fci=fci, wc=wc_calc, opex_bruto=opex_calc,
                                  precio_meoh=p_test,
                                  precio_o2=PRECIO_O2_BASE_USD_T,
                                  credito_inc=CREDITO_INCENTIVO_BASE_USD_T)
        return calcular_van(fc['flujo_libre'], tasa_desc)

    low, high = 100.0, 2500.0
    for _ in range(100):
        mid = (low + high) / 2.0
        val = f_obj(mid)
        if abs(val) < 1e-2:
            return mid
        if val > 0:
            high = mid
        else:
            low = mid
    return mid

## 4. Ejecución del Caso Base y Obtención de Indicadores

Cálculo de los valores oficiales de rentabilidad para el consorcio Solvanta.

In [ ]:
fc_base = construir_flujo_caja()
flujo_libre = fc_base['flujo_libre']

van_10 = calcular_van(flujo_libre, TASA_DESCUENTO_BASE)
van_wacc = calcular_van(flujo_libre, TASA_DESCUENTO_WACC)
tir_base = calcular_tir(flujo_libre)

flujo_ebitda = [-fc_base['tci']] + [fc_base['ebitda'][t] for t in range(1, VIDA_UTIL_ANOS + 1)]
flujo_ebitda[-1] += fc_base['wc']
tir_pretax = calcular_tir(flujo_ebitda)

cum_simple = np.cumsum(flujo_libre)
payback_simple = calcular_payback_exacto(cum_simple)

fcf_desc_10 = [f / ((1.0 + TASA_DESCUENTO_BASE) ** t) for t, f in enumerate(flujo_libre)]
cum_desc_10 = np.cumsum(fcf_desc_10)
payback_desc_10 = calcular_payback_exacto(cum_desc_10)

fcf_desc_wacc = [f / ((1.0 + TASA_DESCUENTO_WACC) ** t) for t, f in enumerate(flujo_libre)]
cum_desc_wacc = np.cumsum(fcf_desc_wacc)
payback_desc_wacc = calcular_payback_exacto(cum_desc_wacc)

mpsp_10 = calcular_mpsp(TASA_DESCUENTO_BASE)
mpsp_wacc = calcular_mpsp(TASA_DESCUENTO_WACC)
costo_unit_opex_bruto = OPEX_BRUTO_BASE / PROD_MEOH_T_ANO
costo_unit_opex_neto = OPEX_NETO_BASE / PROD_MEOH_T_ANO

print(f"VAN (@ 10.0%):           +${van_10:,.2f} USD (+${van_10/1e6:.2f}M USD)")
print(f"VAN (@ 8.5% WACC):       +${van_wacc:,.2f} USD (+${van_wacc/1e6:.2f}M USD)")
print(f"TIR Post-Tax:            {tir_base * 100.0:.2f}%")
print(f"TIR Pre-Tax:             {tir_pretax * 100.0:.2f}%")
print(f"Payback Simple:          {payback_simple:.2f} años")
print(f"Payback Descontado (10%):{payback_desc_10:.2f} años")
print(f"MPSP (@ 10.0%):          ${mpsp_10:.2f} USD/t (${mpsp_10/1000.0:.4f} USD/kg)")
print(f"MPSP (@ 8.5% WACC):      ${mpsp_wacc:.2f} USD/t (${mpsp_wacc/1000.0:.4f} USD/kg)")

## 5. Análisis de Sensibilidad Multivariable (Tornado y Spider)

Evaluación de la resiliencia del proyecto ante desviaciones de ±20% en las variables críticas: tarifa eléctrica, precio de e-metanol, inversión CAPEX, costo de CO₂ y tasa de descuento.

In [ ]:
def evaluar_sensibilidad_van(p_meoh=PRECIO_MEOH_BASE_USD_T, c_elec=COSTO_ELEC_BASE_USD_KWH,
                                fci_val=FCI, c_co2=35.00, tasa_desc=TASA_DESCUENTO_BASE):
    wc_val = fci_val * WC_PCT
    c_elec_anual = POTENCIA_TOTAL_MW * 1000.0 * HORAS_OPERACION_ANO * c_elec
    c_co2_anual = 42194.7 * c_co2
    c_otros = (COSTO_H2O_ANUAL + FONDO_STACKS_SOEC_ANUAL + FONDO_CATALIZADOR_ANUAL +
               MANO_OBRA_ANUAL + fci_val * 0.035)
    opex_b = c_elec_anual + c_co2_anual + c_otros

    fc = construir_flujo_caja(fci=fci_val, wc=wc_val, opex_bruto=opex_b,
                              precio_meoh=p_meoh,
                              precio_o2=PRECIO_O2_BASE_USD_T,
                              credito_inc=CREDITO_INCENTIVO_BASE_USD_T)
    return calcular_van(fc['flujo_libre'], tasa_desc)

tornado_vars = [
    ('Tarifa Eléctrica (USD/kWh)',
     COSTO_ELEC_BASE_USD_KWH * 0.80, COSTO_ELEC_BASE_USD_KWH * 1.20,
     lambda v: evaluar_sensibilidad_van(c_elec=v)),
    ('Precio e-Metanol (USD/t)',
     PRECIO_MEOH_BASE_USD_T * 0.80, PRECIO_MEOH_BASE_USD_T * 1.20,
     lambda v: evaluar_sensibilidad_van(p_meoh=v)),
    ('Inversión CAPEX (FCI)',
     FCI * 0.80, FCI * 1.20,
     lambda v: evaluar_sensibilidad_van(fci_val=v)),
    ('Costo Captura CO2 (USD/t)',
     35.00 * 0.80, 35.00 * 1.20,
     lambda v: evaluar_sensibilidad_van(c_co2=v)),
    ('Tasa Descuento',
     0.08, 0.12,
     lambda v: evaluar_sensibilidad_van(tasa_desc=v))
]

print("Sensibilidad Tornado (±20%):")
for name, v_low, v_high, func in tornado_vars:
    van_low = func(v_low) / 1e6
    van_high = func(v_high) / 1e6
    print(f"{name:<30}: Low = ${van_low:+.2f}M USD | High = ${van_high:+.2f}M USD")

## 6. Generación y Visualización de Gráficos Financieros Profesionales

Generación y exportación de curvas de flujo de caja acumulado, diagrama Tornado y curvas Spider.

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig_out_dir = os.path.join(os.getcwd(), 'figuras_financieras')
os.makedirs(fig_out_dir, exist_ok=True)

# 1. Perfil del Flujo de Caja
fig, ax1 = plt.subplots(figsize=(11, 6))
anos_arr = np.array(fc_base['anos'])
fcf_arr = np.array(fc_base['flujo_libre']) / 1e6
fcf_desc_arr = np.array(fcf_desc_10) / 1e6
cum_desc_arr = np.array(cum_desc_10) / 1e6

colors = ['#C0392B' if x < 0 else '#27AE60' for x in fcf_arr]
ax1.bar(anos_arr - 0.18, fcf_arr, width=0.36, color=colors, alpha=0.85, label='Flujo Libre Anual (M$ USD)')
ax1.bar(anos_arr + 0.18, fcf_desc_arr, width=0.36, color='#2980B9', alpha=0.85, label='Flujo Descontado 10% (M$ USD)')
ax1.set_xlabel('Año de Proyecto', fontweight='bold')
ax1.set_ylabel('Flujo de Caja Anual (M$ USD)', fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(anos_arr, cum_desc_arr, color='#E67E22', linewidth=2.5, marker='o', label='Flujo Acumulado Descontado')
ax2.axhline(0, color='black', linestyle='--', linewidth=1.0)
ax2.set_ylabel('Flujo Acumulado Descontado (M$ USD)', fontweight='bold')
plt.title('Perfil Temporal de Flujo de Caja Libre y Recuperación de Inversión', fontweight='bold', pad=12)
fig.tight_layout()
fig_p1 = os.path.join(fig_out_dir, 'fig_flujo_caja_acumulado.png')
plt.savefig(fig_p1, dpi=300)
plt.close(fig)
print(f"Gráfica guardada: {fig_p1}")

## 7. Reporte Consolidado y Validación de Consistencia Financiera

Resumen global de los estados de resultados proyectados y rentabilidad contractual.

In [ ]:
print("RESUMEN DE RESULTADOS FINANCIEROS RT-03:")
print(f"CAPEX Total (FCI):                  ${FCI/1e6:.2f}M USD")
print(f"Capital de Trabajo (WC):            ${WC/1e6:.2f}M USD")
print(f"Inversión de Capital Total (TCI):   ${TCI/1e6:.2f}M USD")
print(f"OPEX Bruto Anual:                   ${OPEX_BRUTO_BASE/1e6:.2f}M USD/año (${costo_unit_opex_bruto:.2f} USD/t)")
print(f"Crédito por Venta de Oxígeno:       ${CREDITO_O2_ANUAL_BASE/1e6:.2f}M USD/año")
print(f"OPEX Neto Anual:                    ${OPEX_NETO_BASE/1e6:.2f}M USD/año (${costo_unit_opex_neto:.2f} USD/t)")
print(f"VAN (@ 10.0%):                      +${van_10/1e6:.2f}M USD")
print(f"VAN (@ 8.5% WACC):                  +${van_wacc/1e6:.2f}M USD")
print(f"TIR Post-Tax:                       {tir_base*100.0:.2f}%")
print(f"Payback Descontado (10%):           {payback_desc_10:.2f} años")
print(f"Precio Mínimo de Venta (MPSP 10%):  ${mpsp_10:.2f} USD/t")